> y convertirlo a notebook:
>
> ```bash
> pip install jupytext duckdb
> jupytext --to notebook EDA_duckdb.md
> ```

# 🦆 EDA con DuckDB — Soluciones paso a paso (SQL puro)

Este cuaderno crea un dataset sintético en memoria y replica un EDA completo con SQL en DuckDB.  
**Objetivo:** comparar después con NumPy / Pandas / cuDF.

---

## 🔧 Inicialización (Python mínimo para abrir la conexión)

In [2]:
import duckdb
con = duckdb.connect(database=':memory:')

---

## 🧪 Crear dataset sintético en DuckDB (SQL puro)

* 1.000 filas
* Columnas: `ID, Edad, Salario, Experiencia, Horas, Nivel, Satisfaccion`
* `Salario` ~ N(2500, 800²) con ~1% *outliers* (×4)
* `Satisfaccion` ~ N(7, 2²) con ~1.5% `NULL`
* `Horas` ~ N(40, 5²)
* `Nivel` entero 0..4

> Nota: Usamos **Box-Muller** para generar normales desde `random()`.

In [3]:
con.execute(r"""
PRAGMA disable_progress_bar;

-- Limpiar si existe
DROP TABLE IF EXISTS empleados;

-- Crear tabla con base aleatoria y normales via Box-Muller
CREATE TABLE empleados AS
WITH base AS (
  SELECT
    row_number() OVER ()::INTEGER AS id,
    18 + CAST(floor(random()*47) AS INTEGER)             AS edad,          -- 18..64
    -- Normal(0,1) por Box-Muller
    (sqrt(-2*ln(random()))*cos(2*pi()*random()))         AS z1,
    (sqrt(-2*ln(random()))*sin(2*pi()*random()))         AS z2,
    40 + (sqrt(-2*ln(random()))*cos(2*pi()*random()))*5  AS horas_raw,     -- ~ N(40, 5)
    CAST(floor(random()*5) AS INTEGER)                   AS nivel_raw,
    random()                                             AS u_out,
    random()                                             AS u_nan
  FROM range(1000)
),
dens AS (
  SELECT
    id,
    edad,
    -- Salario ~ N(2500, 800)
    (2500 + z1*800) AS salario_base,
    -- Experiencia 0..(edad-18) aproximada (usamos z2 para algo de dispersión)
    GREATEST(0, CAST(edad - (18 + abs(z2*6)) AS INTEGER)) AS experiencia,
    horas_raw AS horas,
    nivel_raw AS nivel,
    u_out,
    u_nan
  FROM base
),
con_out_nan AS (
  SELECT
    id,
    edad,
    -- ~1% outliers: multiplica por 4
    CASE WHEN u_out < 0.01 THEN salario_base*4 ELSE salario_base END AS salario,
    experiencia,
    horas,
    nivel,
    -- Satisfacción ~ N(7,2), con ~1.5% NULL
    CASE WHEN u_nan < 0.015 THEN NULL
         ELSE 7 + (sqrt(-2*ln(random()))*cos(2*pi()*random()))*2
    END AS satisfaccion
  FROM dens
)
SELECT
  id AS ID,
  edad AS Edad,
  salario AS Salario,
  experiencia AS Experiencia,
  horas AS Horas,
  nivel AS Nivel,
  satisfaccion AS Satisfaccion
FROM con_out_nan;
""");

---

## 🧠 Tarea 1 — Descripción básica

In [4]:
print("Filas x Columnas:", con.execute("SELECT COUNT(*) AS filas, COUNT(*) FILTER (WHERE 1=0) AS columnas FROM empleados").fetchall()[0][0], "x 7")
print("Primeras 5:")
print(con.execute("SELECT * FROM empleados LIMIT 5").fetchdf())
print("Esquema:")
print(con.execute("PRAGMA table_info('empleados')").fetchdf())

Filas x Columnas: 1000 x 7
Primeras 5:
   ID  Edad      Salario  Experiencia      Horas  Nivel  Satisfaccion
0   1    56  2495.739283           28  34.453651      1      6.990025
1   2    27  2401.783573            3  43.979668      2      6.456507
2   3    64  2351.579489           46  36.727142      2      6.106670
3   4    37  2190.891403           16  38.023125      1      9.364216
4   5    31  3840.064621            7  44.289476      3      5.359055
Esquema:
   cid          name     type  notnull dflt_value     pk
0    0            ID  INTEGER    False       None  False
1    1          Edad  INTEGER    False       None  False
2    2       Salario   DOUBLE    False       None  False
3    3   Experiencia  INTEGER    False       None  False
4    4         Horas   DOUBLE    False       None  False
5    5         Nivel  INTEGER    False       None  False
6    6  Satisfaccion   DOUBLE    False       None  False


---

## 📏 Tarea 2 — Tipos y rangos

In [5]:
print(con.execute("""
SELECT 
  MIN(Edad) AS min_edad, MAX(Edad) AS max_edad,
  MIN(Salario) AS min_salario, MAX(Salario) AS max_salario,
  AVG(Edad) AS media_edad, MEDIAN(Salario) AS mediana_salario,
  STDDEV_SAMP(Salario) AS std_salario
FROM empleados;
""").fetchdf())

   min_edad  max_edad  min_salario   max_salario  media_edad  mediana_salario  \
0        18        64    46.614328  17004.935014      41.435      2497.955136   

   std_salario  
0   1340.40689  


---

## 💭 Tarea 3 — Filtrado de datos

In [6]:
# 1) Salarios > 4000 €
print(con.execute("SELECT COUNT(*) AS n_altos FROM empleados WHERE Salario > 4000").fetchdf())

# 2) >45h y >10 años experiencia (mostrar IDs)
print(con.execute("""
SELECT ID, Edad, Horas, Experiencia 
FROM empleados 
WHERE Horas > 45 AND Experiencia > 10
ORDER BY Horas DESC
""").fetchdf())

# 3) % nivel >= 3
print(con.execute("""
SELECT 100.0*AVG(CASE WHEN Nivel >= 3 THEN 1 ELSE 0 END) AS pct_nivel_ge3
FROM empleados
""").fetchdf())

   n_altos
0       60
      ID  Edad      Horas  Experiencia
0    985    50  54.821913           30
1     94    40  54.587591           19
2    299    63  53.919632           43
3    180    33  52.632912           15
4    737    58  52.492217           23
..   ...   ...        ...          ...
99   424    53  45.039956           29
100  950    60  45.016044           37
101  667    50  45.011459           29
102  462    41  45.005169           21
103  754    36  45.001122           11

[104 rows x 4 columns]
   pct_nivel_ge3
0           39.1


---

## 🧹 Tarea 4 — Limpieza: valores faltantes (NULL)

In [7]:
# Contar NULL en satisfacción
print(con.execute("SELECT COUNT(*) AS nan_sat FROM empleados WHERE Satisfaccion IS NULL").fetchdf())

# Rellenar NULLs con la media (ignorando NULLs)
con.execute("""
UPDATE empleados
SET Satisfaccion = (SELECT AVG(Satisfaccion) FROM empleados)
WHERE Satisfaccion IS NULL
""");

# Verificar
print(con.execute("SELECT COUNT(*) AS nan_sat FROM empleados WHERE Satisfaccion IS NULL").fetchdf())

   nan_sat
0       23
   nan_sat
0        0


---

## ⚠️ Tarea 5 — Detección de outliers (IQR)

In [8]:
print(con.execute("""
WITH stats AS (
  SELECT 
    quantile_cont(Salario, 0.25) AS q1,
    quantile_cont(Salario, 0.75) AS q3
  FROM empleados
),
bounds AS (
  SELECT q1, q3, (q3-q1) AS iqr,
         q1 - 1.5*(q3-q1) AS lim_inf,
         q3 + 1.5*(q3-q1) AS lim_sup
  FROM stats
)
SELECT 
  (SELECT COUNT(*) FROM empleados, bounds 
   WHERE Salario < bounds.lim_inf OR Salario > bounds.lim_sup) AS n_outliers,
  lim_inf, lim_sup
FROM bounds;
""").fetchdf())

   n_outliers     lim_inf      lim_sup
0          18  238.792027  4803.002671


---

## 🧰 Tarea 6 — Sustitución de outliers (p95)

In [15]:
import time
import pandas as pd

df=con.execute("SELECT * FROM empleados").fetchdf()

t0 = time.time()
altos = df[df["Salario"] > 4000]
cond2 = (df["Horas"] > 45) & (df["Experiencia"] > 10)
sel2 = df.loc[cond2, ["ID","Edad","Horas","Experiencia"]].head()
pct_nivel_ge3 = (df["Nivel"] >= 3).mean() * 100# Calcular p95
t1= time.time()
print("Pandas:")
print("Altos salarios (>4000):", len(altos))
print("Condición 2:")
print(sel2)
print("Pct nivel >=3:", pct_nivel_ge3)
print("Tiempo Pandas: %.6f s" % (t1 - t0))    


t0 = time.time()
print(con.execute("SELECT quantile_cont(Salario, 0.95) AS p95 FROM empleados").fetchdf())
# Reemplazar outliers por p95
con.execute("""
WITH stats AS (
  SELECT quantile_cont(Salario, 0.25) AS q1,
         quantile_cont(Salario, 0.75) AS q3,
         quantile_cont(Salario, 0.95) AS p95
  FROM empleados
),
bounds AS (
  SELECT q1, q3, p95,
         q1 - 1.5*(q3-q1) AS lim_inf,
         q3 + 1.5*(q3-q1) AS lim_sup
  FROM stats
)
UPDATE empleados
SET Salario = (SELECT p95 FROM bounds)
WHERE Salario < (SELECT lim_inf FROM bounds) OR Salario > (SELECT lim_sup FROM bounds);
""");
# Comprobar nuevos extremos
print(con.execute("SELECT MIN(Salario) AS min_salario, MAX(Salario) AS max_salario FROM empleados").fetchdf())
t1= time.time()
print("Tiempo DuckDB: %.6f s" % (t1 - t0))

Pandas:
Altos salarios (>4000): 62
Condición 2:
    ID  Edad      Horas  Experiencia
5    6    47  45.954114           26
11  12    47  46.277216           27
17  18    47  46.709929           21
38  39    54  48.022003           22
52  53    40  47.379764           19
Pct nivel >=3: 39.1
Tiempo Pandas: 0.003635 s
           p95
0  4064.828091
   min_salario  max_salario
0   366.004732  4765.915304
Tiempo DuckDB: 0.015709 s


---

## 📊 Tarea 7 — Agrupaciones y resúmenes

In [ ]:
# Media de salario por nivel
print(con.execute("""
SELECT Nivel, AVG(Salario) AS salario_medio
FROM empleados
GROUP BY Nivel
ORDER BY Nivel
""").fetchdf())

# Satisfacción media por grupo de edad
print(con.execute("""
WITH grupos AS (
  SELECT *,
    CASE 
      WHEN Edad < 30 THEN '<30'
      WHEN Edad <= 50 THEN '30–50'
      ELSE '>50'
    END AS GrupoEdad
  FROM empleados
)
SELECT GrupoEdad, AVG(Satisfaccion) AS sat_media
FROM grupos
GROUP BY GrupoEdad
ORDER BY CASE GrupoEdad WHEN '<30' THEN 1 WHEN '30–50' THEN 2 ELSE 3 END
""").fetchdf())

---

## 🧩 Tarea 8 — Normalización y z-score

In [ ]:
# Normalizar Salario [0,1] y estandarizar Horas (z)
con.execute("ALTER TABLE empleados ADD COLUMN Salario_norm DOUBLE");
con.execute("ALTER TABLE empleados ADD COLUMN Horas_z DOUBLE");

con.execute("""
UPDATE empleados
SET Salario_norm = (Salario - (SELECT MIN(Salario) FROM empleados)) 
                   / ((SELECT MAX(Salario) FROM empleados) - (SELECT MIN(Salario) FROM empleados)),
    Horas_z = (Horas - (SELECT AVG(Horas) FROM empleados)) / (SELECT STDDEV_SAMP(Horas) FROM empleados);
""");

print(con.execute("""
SELECT MIN(Salario_norm) AS min_norm, MAX(Salario_norm) AS max_norm,
       AVG(Horas_z) AS media_z, STDDEV_SAMP(Horas_z) AS std_z
FROM empleados
""").fetchdf())

---

## 🧮 Tarea 9 — Correlaciones

In [ ]:
# Corr(Salario, Experiencia) y Corr(Edad, Satisfaccion)
print(con.execute("""
SELECT 
  corr(Salario, Experiencia) AS corr_sal_exp,
  corr(Edad, Satisfaccion)   AS corr_edad_sat
FROM empleados
""").fetchdf())

---

## 🧾 Tarea 10 — Estadísticas generales por columna

In [ ]:
print(con.execute("""
SELECT 
  'Edad'          AS columna, AVG(Edad) AS media, STDDEV_SAMP(Edad) AS std, MIN(Edad) AS min, MAX(Edad) AS max, 
   100.0*AVG(CASE WHEN Edad IS NULL THEN 1 ELSE 0 END) AS pct_null
UNION ALL
SELECT 'Salario', AVG(Salario), STDDEV_SAMP(Salario), MIN(Salario), MAX(Salario),
   100.0*AVG(CASE WHEN Salario IS NULL THEN 1 ELSE 0 END)
FROM empleados
UNION ALL
SELECT 'Experiencia', AVG(Experiencia), STDDEV_SAMP(Experiencia), MIN(Experiencia), MAX(Experiencia),
   100.0*AVG(CASE WHEN Experiencia IS NULL THEN 1 ELSE 0 END)
FROM empleados
UNION ALL
SELECT 'Horas', AVG(Horas), STDDEV_SAMP(Horas), MIN(Horas), MAX(Horas),
   100.0*AVG(CASE WHEN Horas IS NULL THEN 1 ELSE 0 END)
FROM empleados
UNION ALL
SELECT 'Nivel', AVG(Nivel), STDDEV_SAMP(Nivel), MIN(Nivel), MAX(Nivel),
   100.0*AVG(CASE WHEN Nivel IS NULL THEN 1 ELSE 0 END)
FROM empleados
UNION ALL
SELECT 'Satisfaccion', AVG(Satisfaccion), STDDEV_SAMP(Satisfaccion), MIN(Satisfaccion), MAX(Satisfaccion),
   100.0*AVG(CASE WHEN Satisfaccion IS NULL THEN 1 ELSE 0 END)
FROM empleados
ORDER BY columna;
""").fetchdf())

---

## ✅ Conclusión

Has realizado un EDA completo **usando solo SQL en DuckDB**:

* Generación del dataset (normales, `NULL`, *outliers*)
* Descriptivos, filtrado, limpieza (`UPDATE ... SET ... WHERE`)
* IQR y *winsorizing* (reemplazo por p95)
* Agrupaciones, normalización, correlaciones

**Siguiente paso** (en cuadernos aparte):

* Comparativa **NumPy vs Pandas vs DuckDB** (líneas, legibilidad, ergonomía)
* EDA con **cuDF** (RAPIDS) para GPU

```

---
